# 🔄 12 · Aligning subtomograms fast: constrained correlation and rotational search
### Why the missing wedge fools ordinary correlation, how to fix it, and how to try all rotations at once

**BiologyToResearch** · Part C — Classical computational cryo-ET (2009–2019) · Module 12 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/12_fast_subtomogram_alignment.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 3–4 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, Matplotlib |
| **Prerequisites** | Modules 02 (rotations), 03 (Fourier transform, phase vs magnitude, cross-correlation), 07 (missing wedge) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | **Xu, Beck & Alber 2012** (J Struct Biol — fast constrained alignment; the centrepiece); cited foundations: Förster *et al.* 2008 (constrained correlation), Kovacs & Wriggers 2002 (fast rotational matching) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments here are small, controlled *toy versions* of published methods. They illustrate the core idea of the
> paper; they do **not** reproduce its numbers (we use 2D slices, 32³ volumes, a coarse rotation grid instead of the
> paper's spherical-harmonic FFT, and our own noise model). **Nothing is stored in the repository:** molecules are
> real Protein Data Bank (PDB) models, downloaded the first time a notebook needs them and cached on your computer
> (offline, code-built stand-ins of the same size and symmetry are used instead); all images and data are simulated by
> code.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** subtomogram alignment as a search over 6 degrees of freedom and **estimate** the cost of brute-force search.
2. **Demonstrate** that ordinary correlation of two missing-wedge subtomograms prefers rotations that *align the wedges*
   rather than the molecules.
3. **Implement** the constrained correlation of Xu, Beck & Alber (2012), eqs. 1–3: compare only Fourier coefficients
   observed in **both** subtomograms, after removing the means.
4. **Derive and verify** the translation-free rotational score $c'(R)=c_0/\sqrt{c_1c_2}$ built from Fourier magnitudes,
   and its three ingredients $p_i,q_i$.
5. **Implement** an exact "all angles at once" rotational search in 2D with polar resampling and one FFT, and **show** it
   equals brute force but is much faster.
6. **Build** a 3D aligner (rotation grid scored by $c'$ → candidates → FFT translation search) and **measure** its
   accuracy versus noise and tilt range, and its speed versus brute force.
7. **Describe** how the paper does the 3D search exactly with spherical harmonics and an FFT over rotations.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, molecules | 5 min |
| 1 · The alignment problem | 6 degrees of freedom; how slow is brute force? | 15 min |
| 2 · The missing wedge fools correlation | a 2D tomogram-slice experiment | 20 min |
| 3 · Constrained correlation | eqs. 1–3, implemented and tested | 25 min |
| 4 · Separating rotation from translation | magnitudes, $c'(R)$, the $p_i/q_i$ table | 20 min |
| 5 · All angles at once (2D) | polar rings + FFT = exact fast rotational search | 25 min |
| 6 · Fast alignment in 3D | rotation grid + $c'$ + candidates + translation; accuracy & speed | 30 min |
| 📄 / ✅ / 🧠 / 🚀 | paper connection, checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

A cellular tomogram may contain thousands of ribosomes, each pointing in a different direction. Each one, cut out as a
**subtomogram**, is far too noisy (SNR ≈ 0.01–0.1) to show anything but a blob. Averaging many copies would reveal their
structure — the large and small subunits, a bound tRNA, a nascent chain in the exit tunnel, a membrane anchor — but only if
every copy is first **turned and shifted onto the same reference frame**. Two things make this hard: the search space is
huge (every rotation × every shift), and each subtomogram is missing a *different* wedge of information because the
microscope could not tilt beyond ±60°. In 2012, Xu, Beck & Alber showed how to make this search both **wedge-aware** and
**thousands of times faster**, enabling reference-free classification of large subtomogram sets.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from scipy.spatial.transform import Rotation

SEED = 0
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

Copied verbatim from [`course_toolkit.py`](../course_toolkit.py): molecule building (Module 00), rotations and the ZYZ
Euler convention (Module 02), Fourier helpers (Module 03) and the missing-wedge mask (Module 07).

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def rot_x(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def rot_y(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def euler_zyz_to_matrix(phi, theta, psi):
    """R = Rz(phi) · Ry(theta) · Rz(psi)  (radians, acting on (x, y, z) vectors)."""
    return rot_z(phi) @ rot_y(theta) @ rot_z(psi)


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


def rotation_angle_deg(R1, R2) -> float:
    """Geodesic distance between two rotations: arccos((tr(R1ᵀR2) − 1)/2), in degrees."""
    c = (np.trace(R1.T @ R2) - 1) / 2
    return float(np.degrees(np.arccos(np.clip(c, -1, 1))))


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def show_orthoslices(vol, title: str = "", axes=None, cmap: str = "gray"):
    """Show the three central orthogonal slices (xy, xz, yz) of a volume."""
    import matplotlib.pyplot as plt
    if axes is None:
        _, axes = plt.subplots(1, 3, figsize=(7.5, 2.6))
    z, y, x = (np.array(vol.shape) // 2)
    for ax, im, lab in zip(axes, [vol[z], vol[:, y, :], vol[:, :, x]], ["xy (z mid)", "xz (y mid)", "yz (x mid)"]):
        ax.imshow(im, cmap=cmap)
        ax.set_title(f"{title} {lab}".strip(), fontsize=8)
        ax.axis("off")
    return axes
print("🧰 toolkit loaded")

In [ ]:
# ── Data: the 70S ribosome (PDB 4V4R) ──
raw_rib, voxel_A, source = zoo_map("4v4r")
rib40 = prepare_map(raw_rib, size=40, sigma=1.0)       # for the 2D slices
rib32 = prepare_map(raw_rib, size=32, sigma=1.0)       # for 3D alignment (32³ voxels of 1 nm)
print(f"4V4R 70S ribosome: raw {raw_rib.shape} at {voxel_A:.0f} Å/voxel ← {source}")

---
## 1 · The alignment problem

> 📍 **Where we are.** Module 02 taught rotations (ZYZ Euler angles, rotation matrices, the geodesic angle between two
> rotations); Module 03 showed that a **shift changes only the phases** of the Fourier transform and that cross-correlation
> for *all* shifts costs one FFT; Module 07 explained the **missing wedge**; Module 11 found particles. This module aligns
> them — the step that makes averaging, classification and discovery possible (Modules 13–15).

Aligning a subtomogram $g$ to a subtomogram (or reference) $f$ means finding the rigid transform — a rotation $R$ (3
angles) and a translation $\mathbf a$ (3 shifts) — that makes the transformed $g$ look most like $f$:

$$(\hat R,\hat{\mathbf a})=\arg\max_{R,\mathbf a}\ \text{similarity}\big(f,\ \tau_{\mathbf a}\Lambda_R\,g\big),
\qquad (\Lambda_R g)(\mathbf x)=g(R^{-1}\mathbf x),\quad (\tau_{\mathbf a}g)(\mathbf x)=g(\mathbf x-\mathbf a).$$

*In words:* rotate $g$ by $R$ ($\Lambda_R$), shift it by $\mathbf a$ ($\tau_{\mathbf a}$), compare with $f$, and keep the
best. That is **6 degrees of freedom**. Thanks to the FFT, all shifts are tried at once for a given rotation (Module 03),
so the cost is (number of rotations) × (one rotation + one FFT correlation). How many rotations are needed? A uniform grid
with angular step $\Delta$ (radians) has about $8\pi^2/\Delta^3$ rotations (the "volume" of the rotation group divided by
the volume of one cell).

In [ ]:
# ── 1.1 Rotation grids and the cost of brute force ──
def euler_grid(step_deg):
    """Near-uniform ZYZ grid: θ in steps Δ; φ with ≈ 360·sin θ/Δ samples (fewer near the poles); ψ in steps Δ."""
    Rs = []
    for th in np.arange(0, 180 + 1e-9, step_deg):
        n_phi = max(1, int(round(360 * np.sin(np.radians(th)) / step_deg)))
        for phi in np.arange(n_phi) * 360 / n_phi:
            for psi in np.arange(0, 360 - 1e-9, step_deg):
                Rs.append(euler_zyz_to_matrix(np.radians(phi), np.radians(th), np.radians(psi)))
    return np.array(Rs)


def euler_grid_size(step_deg):
    """Number of rotations euler_grid(step_deg) would contain (without building them)."""
    n_psi = len(np.arange(0, 360 - 1e-9, step_deg))
    return sum(max(1, int(round(360 * np.sin(np.radians(th)) / step_deg))) * n_psi
               for th in np.arange(0, 180 + 1e-9, step_deg))


def time_one_rotation(n, repeats=5):
    """Seconds for one brute-force step on an n³ volume: rotate + FFT-based correlation over all shifts."""
    v = rng.normal(size=(n, n, n)).astype(np.float32)
    R = random_rotation(rng)
    t0 = time.time()
    for _ in range(repeats):
        vr = rotate_volume(v, R)
        np.real(np.fft.ifftn(np.fft.fftn(v) * np.conj(np.fft.fftn(vr))))
    return (time.time() - t0) / repeats


t32, t64 = time_one_rotation(32), time_one_rotation(64, repeats=2)
print(f"one rotation + translation search: {1000 * t32:.1f} ms at 32³, {1000 * t64:.0f} ms at 64³\n")
print(f"{'step Δ':>7s} | {'# rotations':>12s} | {'8π²/Δ³':>9s} | {'brute force at 32³':>18s} | {'at 64³':>12s}")
steps = [30, 20, 15, 10, 5, 2.5]
sizes = [euler_grid_size(s) for s in steps]
for s, nr in zip(steps, sizes):
    est = 8 * np.pi**2 / np.radians(s)**3
    print(f"{s:6.1f}° | {nr:12,d} | {est:9,.0f} | {nr * t32 / 60:15.1f} min | {nr * t64 / 3600:9.1f} h")

🔍 **What just happened?** Each halving of the angular step multiplies the number of rotations by 8. At the 2.5° sampling
used in the paper's speed benchmark, a brute-force search over 64³ volumes takes hours to days per *pair* on one core —
and reference-free classification needs **all pairs** of thousands of subtomograms (Module 14). The paper reports 153 s
instead of ≈ 9.67 × 10⁵ s (more than 10 days) for exhaustive scanning of 64³ volumes at 2.5° in MATLAB, roughly 6,000×
faster. Two ideas make that possible; this notebook builds both. But first, a more basic problem: *what* should we
maximize?

---
## 2 · The missing wedge fools ordinary correlation

🧫 Every subtomogram is cut from a tomogram reconstructed from a tilt series with a limited range (±60° here). In Fourier
space, a **wedge** of coefficients is never measured (Module 07): the mask is $M(\xi)=\mathbb 1\big(|\xi_z|\le|\xi_x|
\tan\theta_{\max}\big)$, fixed in the **microscope's** frame. Two copies of the ribosome lying in different orientations
are therefore missing *different* parts of *their own* structure.

**A 2D laboratory.** For a single-axis tilt series about $y$, every $x$–$z$ slice of the tomogram is reconstructed
independently from 1D projections, with its own 2D missing wedge. So a 2D image in the $(x,z)$ plane, rotated *within*
that plane, is an exact miniature of the 3D problem — and 100× cheaper. Our 2D "molecule" is a 6-nm-thick central slab
of the ribosome. When we simulate an observation we rotate it, remove the wedge, add noise, and remove the wedge again
(in a real reconstruction there is no noise in the wedge either, because nothing was measured there).

In [ ]:
# ── 2.1 Helpers for 2D tomogram slices with a missing wedge ──
N2 = 64
slab = rib40[:, 17:23, :].sum(axis=1)                    # 6-nm slab through the ribosome, an [z, x] image
OBJ2 = np.zeros((N2, N2), np.float32)
OBJ2[12:52, 12:52] = slab
OBJ2 /= OBJ2.max()


def rot2d(a):
    """2×2 rotation matrix acting on (x, z) vectors (same convention as rotate_image)."""
    return np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])


def wedge_mask_rotated(shape, R, tilt_max_deg=60.0):
    """Missing-wedge mask of an object that was rotated by R after imaging: (Λ_R M)(ξ) = M(Rᵀξ).

    Works for 2D images [z, x] (R is 2×2 on (x, z)) and 3D volumes [z, y, x] (R is 3×3 on (x, y, z)).
    """
    k = freq_grid(shape)
    vec = np.stack([k[1], k[0]]) if len(shape) == 2 else np.stack([k[2], k[1], k[0]])
    kp = np.tensordot(R.T, vec, axes=1)                  # Rᵀξ, component by component
    m = np.abs(kp[-1]) <= np.abs(kp[0]) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True
    return m.astype(np.float32)


def observe(obj, R, shift, snr, tilt_max_deg, rng, rotate=rotate_volume):
    """Rotate/shift an object, then image it: missing wedge → noise (SNR = var ratio) → missing wedge again.

    Returns (observation, its Fourier mask). snr=None means noise-free; tilt_max_deg=None means no missing wedge.
    """
    x = rotate(obj, R, shift)
    W = np.ones(x.shape, np.float32) if tilt_max_deg is None else missing_wedge_mask(x.shape, tilt_max_deg)
    s = apply_fourier_mask(x, W)
    if snr is None:
        return s, W
    noise = rng.normal(0, np.sqrt(s.var() / snr), x.shape)
    return apply_fourier_mask(s + noise, W), W


def observe_2d(angle, shift, snr, tilt_max_deg, rng):
    """2D version: rotate OBJ2 by `angle` (radians) in the x–z plane and image it."""
    return observe(OBJ2, angle, shift, snr, tilt_max_deg, rng, rotate=rotate_image)


assert np.array_equal(wedge_mask_rotated((N2, N2), np.eye(2)), missing_wedge_mask((N2, N2))), "mask convention"
A_F, A_G = 20.0, 145.0                                    # orientations (°) of our example pair
TRUE_2D = (A_F - A_G) % 360                               # rotating g by this angle superimposes it on f
f2, M2 = observe_2d(np.radians(A_F), (0, 0), None, 60, rng)
g2, _ = observe_2d(np.radians(A_G), (0, 0), None, 60, rng)
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
axes[0].imshow(OBJ2); axes[0].set_title("the 2D 'molecule'\n(ribosome slab)")
axes[1].imshow(f2); axes[1].set_title(f"f: rotated {A_F:.0f}°,\nwedge removed")
axes[2].imshow(g2); axes[2].set_title(f"g: rotated {A_G:.0f}°,\nwedge removed")
axes[3].imshow(np.log1p(np.fft.fftshift(np.abs(np.fft.fft2(f2)))), cmap="magma"); axes[3].set_title("|F f| (log)")
axes[4].imshow(np.log1p(np.fft.fftshift(np.abs(np.fft.fft2(g2)))), cmap="magma"); axes[4].set_title("|F g| (log)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
axes[0].set_ylabel("z"); axes[0].set_xlabel("x")
plt.tight_layout(); plt.show()

🔍 The two observations are the *same* molecule, but each is smeared vertically (along $z$, the beam direction) and their
spectra are both missing the same vertical double-cone **in the microscope frame**. After rotating $g$ by 235° (= −125°)
to match $f$, $g$'s blank wedge is rotated too — it no longer coincides with $f$'s.

### 2.2 Experiment: what does plain correlation prefer?

For 30 random pairs (noise-free!) we rotate $g$ by every angle α in 3° steps, compute the ordinary normalized correlation
with $f$ (best over all shifts), and record the winning α. If plain correlation worked, the winning α would track the
true angle. Let us see.

In [ ]:
# ── 2.2 Plain correlation over all angles, 30 noise-free pairs ──
K2 = radial_frequency((N2, N2))
BAND2 = (K2 <= 0.25).astype(np.float32)                  # use frequencies up to 1/(4 nm), where the signal is


def corr_all_shifts(f, g, omega):
    """Normalized correlation between f and g over all cyclic shifts, using only Fourier coefficients where omega = 1
    (DC removed). Returns (best score, best shift in index order)."""
    F, G = np.fft.fftn(f) * omega, np.fft.fftn(g) * omega
    F.flat[0] = G.flat[0] = 0
    norm = np.sqrt(np.sum(np.abs(F)**2) * np.sum(np.abs(G)**2))
    if norm == 0:                                        # no jointly observed coefficient at all
        return -np.inf, np.zeros(f.ndim, int)
    cmap = np.real(np.fft.ifftn(F * np.conj(G))) * F.size / norm
    i = np.unravel_index(np.argmax(cmap), cmap.shape)
    n = np.array(cmap.shape)
    return float(cmap[i]), (np.array(i) + n // 2) % n - n // 2


ANG = np.arange(0, 360, 3)
rng_pairs = np.random.default_rng(3)
pairs = []                                               # (f, g, true angle in degrees)
for _ in range(30):
    a_f, a_g = rng_pairs.uniform(0, 2 * np.pi, 2)
    f, _ = observe_2d(a_f, (0, 0), None, 60, rng_pairs)
    g, _ = observe_2d(a_g, (0, 0), None, 60, rng_pairs)
    pairs.append((f, g, np.degrees(a_f - a_g) % 360))
t0 = time.time()
plain_curves = np.array([[corr_all_shifts(f, rotate_image(g, np.radians(a)), BAND2)[0] for a in ANG]
                         for f, g, _ in pairs])
plain_best = ANG[np.argmax(plain_curves, axis=1)]
true_ang = np.array([t for _, _, t in pairs])


def ang_err(a, b):
    """Smallest difference between two angles in degrees."""
    d = np.abs(np.asarray(a) - np.asarray(b)) % 360
    return np.minimum(d, 360 - d)


print(f"30 pairs × {len(ANG)} angles in {time.time() - t0:.1f} s")
print("winning angle (mod 180°):", np.sort(plain_best % 180))
print(f"median error vs the TRUE angle: {np.median(ang_err(plain_best, true_ang)):.0f}°")
# control: the same code on 10 pairs imaged WITHOUT a missing wedge
ctrl_err = []
for _ in range(10):
    a_f, a_g = rng_pairs.uniform(0, 2 * np.pi, 2)
    f, _ = observe_2d(a_f, (0, 0), None, None, rng_pairs)
    g, _ = observe_2d(a_g, (0, 0), None, None, rng_pairs)
    best = ANG[np.argmax([corr_all_shifts(f, rotate_image(g, np.radians(a)), BAND2)[0] for a in ANG])]
    ctrl_err.append(ang_err(best, np.degrees(a_f - a_g)))
print(f"control without a missing wedge: median error {np.median(ctrl_err):.1f}°")

In [ ]:
# ── 2.3 Plot: plain correlation chases the wedge ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
k = 0
axes[0].plot(ANG, plain_curves[k], color="tab:gray", label="plain correlation")
axes[0].axvline(true_ang[k], color="tab:green", lw=2, label=f"true angle ({true_ang[k]:.0f}°)")
for a in (0, 180, 360):
    axes[0].axvline(a, color="tab:red", ls=":", lw=1.5)
axes[0].set_xlabel("rotation α applied to g (°)"); axes[0].set_ylabel("best correlation over shifts")
axes[0].set_title("one pair: red dotted = wedges aligned (0°, 180°)"); axes[0].legend(fontsize=7)
axes[1].scatter(true_ang, plain_best, color="tab:gray", s=18, label="plain correlation's choice")
axes[1].plot([0, 360], [0, 360], color="tab:green", lw=1, label="perfect alignment")
axes[1].set_xlabel("true angle (°)"); axes[1].set_ylabel("chosen angle (°)"); axes[1].set_title("30 noise-free pairs")
axes[1].set_yticks([0, 90, 180, 270, 360]); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Even **without any noise**, ordinary correlation almost always picks α ≈ 0° or 180° — the two
rotations that put $g$'s missing wedge exactly on top of $f$'s — whatever the true angle is. The reason: the wedge leaves a
strong, shared *artefact pattern* (elongation and streaks along $z$, missing structure in the same directions), and
aligning the artefacts gives a higher correlation than aligning the molecules. The control run proves the wedge is the
culprit: without a missing wedge, the very same code finds the true angle (0° and 180° are also the only rotations that
need no pixel interpolation, but that alone does not create the bias).

⚠️ This is not a toy curiosity: averaging subtomograms aligned this way produces a reference that *looks* plausible but is
dominated by the wedge — one reason early subtomogram-averaging results needed great care.

✏️ **Your turn.** In cell 2.2 replace `BAND2` by `BAND2 * 0 + 1` (use all frequencies) or the `60` tilt by `80`. Does the
bias disappear? (It shrinks as the wedge narrows, but only vanishes with no wedge at all.)

---
## 3 · Constrained correlation: compare only what both subtomograms measured

The fix, introduced by Förster *et al.* (2008) and formulated in Fourier space by **Xu, Beck & Alber (2012)**, is to
compare $f$ and the rotated $g$ **only on Fourier coefficients that were observed in both**. The rotated $g$ carries its
wedge with it, so the jointly valid set is

$$\Omega = M_f\cdot\Lambda_R M_g,\qquad (\Lambda_R M_g)(\xi)=M_g(R^{-1}\xi).$$

Then both volumes are filtered to $\Omega$ (eqs. 1–2 of the paper):

$$\tilde f=\Re\big\{\mathcal F^{-1}\big[(\mathcal Ff)\,\Omega\big]\big\},\qquad
\tilde g=\Re\big\{\mathcal F^{-1}\big[(\mathcal F\,\tau_{\mathbf a}\Lambda_R g)\,\Omega\big]\big\},$$

and compared with a mean-normalized (Pearson) correlation (eq. 3):

$$c(R,\mathbf a)=\frac{\sum_{\mathbf x}\big[\tilde f(\mathbf x)-\mu(\tilde f)\big]\big[\tilde g(\mathbf x)-\mu(\tilde g)\big]}
{\sqrt{\sum_{\mathbf x}\big[\tilde f(\mathbf x)-\mu(\tilde f)\big]^2\ \sum_{\mathbf x}\big[\tilde g(\mathbf x)-\mu(\tilde g)\big]^2}} .$$

*In words:* blank out, in **both** volumes, everything either of them failed to measure; then ask how similar what remains
is. Neither wedge can now create a shared artefact, because both volumes are missing exactly the same coefficients.
Subtracting the means and dividing by the spreads (Pearson) makes the score insensitive to overall brightness and contrast.

🧮 **Parseval's theorem** says that sums of products in real space equal sums of products of Fourier coefficients (up to
a constant). Removing the mean is the same as zeroing the $\xi=0$ (DC) coefficient. So eq. 3 can be computed in Fourier
space, and — because a shift is a phase factor $e^{-2\pi i\,\mathbf a\cdot\xi}$ — **for all shifts $\mathbf a$ at once**
with one inverse FFT (the paper's eq. 7, which AITom's `cons_corr_max` implements):

$$c(R,\mathbf a)=\frac{\sum_\xi (\mathcal Ff)(\xi)\,\overline{(\mathcal F\Lambda_Rg)(\xi)}\,\Omega(\xi)\,e^{2\pi i\,\mathbf a\cdot\xi}}
{\sqrt{\sum_\xi|\mathcal Ff|^2\Omega\ \sum_\xi|\mathcal F\Lambda_Rg|^2\Omega}}\qquad(\xi\neq0).$$

In [ ]:
# ── 3.1 Constrained correlation, exactly as eqs. 1–3, and its all-shifts Fourier form ──
def constrained_correlation(f, Mf, g_rot, Mg_rot):
    """Eqs. 1–3 of Xu, Beck & Alber (2012) at zero shift: filter both to Ω = M_f · Λ_R M_g, then Pearson."""
    omega = Mf * Mg_rot
    f_t = np.real(np.fft.ifftn(np.fft.fftn(f) * omega))
    g_t = np.real(np.fft.ifftn(np.fft.fftn(g_rot) * omega))
    return pearson(f_t, g_t)


def constrained_correlation_shifts(f, Mf, g_rot, Mg_rot, band=1.0):
    """Same score for every cyclic shift at once (Fourier form). Returns (best score, best shift [index order])."""
    return corr_all_shifts(f, g_rot, Mf * Mg_rot * band)


# check: both forms agree at zero shift (here the best shift is 0 because f and g are centred)
a_test = np.radians(TRUE_2D)                                   # the true angle for the pair f2, g2 above
g2_rot, M2_rot = rotate_image(g2, a_test), wedge_mask_rotated((N2, N2), rot2d(a_test))
c_real = constrained_correlation(f2, M2, g2_rot, M2_rot)
c_four, sh = constrained_correlation_shifts(f2, M2, g2_rot, M2_rot)
print(f"eqs. 1–3 in real space: {c_real:.5f} | Fourier form, best over shifts: {c_four:.5f} at shift {sh}")
print(f"plain Pearson of the same two images: {pearson(f2, g2_rot):.3f}")
assert abs(c_real - c_four) < 1e-4 and np.all(sh == 0), "the two forms of constrained correlation should agree"

In [ ]:
# ── 3.2 Re-run the 30 pairs with constrained correlation ──
t0 = time.time()
cons_curves = np.array([[constrained_correlation_shifts(f, M2, rotate_image(g, np.radians(a)),
                                                        wedge_mask_rotated((N2, N2), rot2d(np.radians(a))), BAND2)[0]
                         for a in ANG] for f, g, _ in pairs])
cons_best = ANG[np.argmax(cons_curves, axis=1)]
print(f"done in {time.time() - t0:.1f} s")
print(f"median error — plain: {np.median(ang_err(plain_best, true_ang)):.0f}°, "
      f"constrained: {np.median(ang_err(cons_best, true_ang)):.1f}°")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].plot(ANG, plain_curves[0] / plain_curves[0].max(), color="tab:gray", label="plain (scaled)")
axes[0].plot(ANG, cons_curves[0] / cons_curves[0].max(), color="tab:blue", label="constrained (scaled)")
axes[0].axvline(true_ang[0], color="tab:green", lw=2, label="true angle")
axes[0].set_xlabel("rotation α applied to g (°)"); axes[0].set_ylabel("score / max"); axes[0].legend(fontsize=7)
axes[0].set_title("same pair as before")
axes[1].scatter(true_ang, plain_best, color="tab:gray", s=16, label="plain")
axes[1].scatter(true_ang, cons_best, color="tab:blue", s=16, label="constrained")
axes[1].plot([0, 360], [0, 360], color="tab:green", lw=1)
axes[1].set_xlabel("true angle (°)"); axes[1].set_ylabel("chosen angle (°)"); axes[1].legend(fontsize=7)
axes[1].set_title("30 noise-free pairs")
plt.tight_layout(); plt.show()
assert np.median(ang_err(cons_best, true_ang)) < 10, "constrained correlation should recover the true angles"

🔍 **What just happened?** Restricting the comparison to the jointly observed coefficients removes the wedge bias:
the constrained score peaks at the true angle, and the chosen angles now lie on the diagonal. (The small residual errors
come from interpolation and the 3° step.) The same holds in 3D, as the next cell checks on the ribosome.

In [ ]:
# ── 3.3 The same bias in 3D (ribosome, 32³, noise-free): true rotation vs "wedge-aligned" rotations ──
WEDGE_KEEPERS = [np.eye(3), rot_x(np.pi), rot_y(np.pi), rot_z(np.pi)]   # rotations that map the wedge onto itself
rng3 = np.random.default_rng(11)
wins = {"plain": 0, "constrained": 0}
for trial in range(8):
    R_f, R_g = random_rotation(rng3), random_rotation(rng3)
    f3, Mf3 = observe(rib32, R_f, (0, 0, 0), None, 60, rng3)
    g3, _ = observe(rib32, R_g, (0, 0, 0), None, 60, rng3)
    R_true = R_f @ R_g.T
    score_plain = lambda R: pearson(f3, rotate_volume(g3, R))
    score_cons = lambda R: constrained_correlation(f3, Mf3, rotate_volume(g3, R), wedge_mask_rotated(f3.shape, R))
    for name, score in [("plain", score_plain), ("constrained", score_cons)]:
        wins[name] += score(R_true) > max(score(R) for R in WEDGE_KEEPERS)
print("pairs (of 8) in which the TRUE rotation beats every wedge-aligning rotation:", wins)

🔍 In 3D, too, plain correlation often prefers a rotation that superimposes the two wedges (identity or a 180° turn about
the $x$, $y$ or $z$ axis of the microscope) over the true rotation; constrained correlation does not.

---
## 4 · Separating rotation from translation: the phase-free score $c'(R)$

Constrained correlation tells us *what* to maximize. Now speed. For a given rotation the FFT handles all shifts; the
remaining problem is the rotation search. Module 03's key fact helps: **shifting a volume changes only the phases of its
Fourier coefficients, not their magnitudes**, $|\mathcal F\tau_{\mathbf a}g|=|\mathcal Fg|$. So if we compare only
**magnitudes**, the unknown shift drops out and we can search rotations alone. Replacing $\mathcal Ff\cdot\overline{\mathcal
F\Lambda_Rg}$ in the numerator by $|\mathcal Ff|\cdot|\mathcal F\Lambda_Rg|$ gives the paper's rotational score (eqs. 8–9,
written here with squared masks so that it also works for weighted, non-binary masks — as in AITom):

$$c'(R)=\frac{\sum_\xi |\mathcal Ff|M_f^2\ \Lambda_R\big[|\mathcal Fg|M_g^2\big]}
{\sqrt{\sum_\xi|\mathcal Ff|^2M_f^2\ \Lambda_R[M_g^2]\ \cdot\ \sum_\xi M_f^2\ \Lambda_R\big[|\mathcal Fg|^2M_g^2\big]}}
=\frac{c_0(R)}{\sqrt{c_1(R)\,c_2(R)}},\qquad c_i(R)=\sum_\xi p_i\ \Lambda_R q_i .$$

| $i$ | $p_i$ (from $f$, fixed) | $q_i$ (from $g$, rotated) | meaning |
|---|---|---|---|
| 0 | $\lvert\mathcal Ff\rvert M_f^2$ | $\lvert\mathcal Fg\rvert M_g^2$ | overlap of the two magnitude spectra |
| 1 | $\lvert\mathcal Ff\rvert^2M_f^2$ | $M_g^2$ | energy of $f$ on the jointly observed region |
| 2 | $M_f^2$ | $\lvert\mathcal Fg\rvert^2M_g^2$ | energy of $g$ on the jointly observed region |

*In words:* $c'$ is a normalized correlation of the two **magnitude spectra**, restricted to the jointly observed region,
exactly mirroring the constrained correlation — but blind to shifts. Each $c_i$ has the form "fixed array × rotated array,
summed": a **rotational correlation**, which is what Section 5 learns to compute for all rotations at once.

💡 **An upper bound.** For any shift $\mathbf a$, $|\sum A\,\bar B\,e^{i\phi}|\le\sum|A||B|$ (the triangle inequality),
so $c'(R)\ge c(R,\mathbf a)$: the magnitude score is an optimistic bound of the best constrained correlation at that
rotation. Good rotations must have high $c'$ — but a high $c'$ is not proof, because phases (which carry the "where",
and hence the handedness and arrangement of the parts) were thrown away. The paper therefore keeps **several** high-$c'$
rotations as candidates and decides among them with the full $c(R,\mathbf a)$.

In [ ]:
# ── 4.1 Check: magnitudes ignore shifts; c'(α) ≥ max_a c(α, a) for every angle ──
g_shift = np.roll(g2, (5, -3), axis=(0, 1))
print("max |  |F g| − |F shifted g|  | / max|F g| =",
      f"{np.abs(np.abs(np.fft.fft2(g2)) - np.abs(np.fft.fft2(g_shift))).max() / np.abs(np.fft.fft2(g2)).max():.2e}")


def cprime_exact(f, Mf, g_rot, Mg_rot, band=1.0):
    """c' for one rotation, from the already-rotated g and its rotated mask (binary masks: M² = M)."""
    Af, Ag = np.abs(np.fft.fftn(f)), np.abs(np.fft.fftn(g_rot))
    omega = Mf * Mg_rot * band
    omega.flat[0] = 0
    return float(np.sum(Af * Ag * omega) / np.sqrt(np.sum(Af**2 * omega) * np.sum(Ag**2 * omega)))


f_n, Mf_n = observe_2d(np.radians(A_F), (2, -1), 1.0, 60, rng)       # SNR 1, f shifted
g_n, _ = observe_2d(np.radians(A_G), (-1, 3), 1.0, 60, rng)          # g shifted differently
cp_curve, c_curve = [], []
for a in ANG:
    gr, Mr = rotate_image(g_n, np.radians(a)), wedge_mask_rotated((N2, N2), rot2d(np.radians(a)))
    cp_curve.append(cprime_exact(f_n, Mf_n, gr, Mr, BAND2))
    c_curve.append(constrained_correlation_shifts(f_n, Mf_n, gr, Mr, BAND2)[0])
cp_curve, c_curve = np.array(cp_curve), np.array(c_curve)
print(f"c' ≥ max_a c at all {len(ANG)} angles: {np.all(cp_curve >= c_curve - 1e-6)}")
fig, ax = plt.subplots(figsize=(6.6, 3.2))
ax.plot(ANG, cp_curve, color="tab:orange", label="c'(α): magnitudes only (shift-free)")
ax.plot(ANG, c_curve, color="tab:blue", label="max over shifts of c(α, a): full constrained correlation")
ax.axvline(TRUE_2D, color="tab:green", lw=2, label=f"true angle ({TRUE_2D:.0f}°)")
ax.axvline((TRUE_2D + 180) % 360, color="tab:green", ls="--", lw=1, label="true angle ± 180°")
ax.set_xlabel("rotation α applied to g (°)"); ax.set_ylabel("score"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("The magnitude score bounds the full score from above (SNR 1)")
plt.tight_layout(); plt.show()
assert np.all(cp_curve >= c_curve - 1e-6), "c' must be an upper bound of c"

🔍 **What just happened?** The orange curve never falls below the blue one, and both peak near the true angle. But the
orange curve has a **second, equally high peak 180° away**: a magnitude spectrum $|\mathcal F|$ of a real image is
symmetric under $\xi\to-\xi$, and in 2D that is exactly a 180° rotation — so $c'$ cannot distinguish α from α + 180°. The
phases can (blue curve). In 3D, $\xi\to-\xi$ is not a rotation (it is a point reflection), but for roughly
ellipsoidal, low-resolution molecules it creates near-ties between a rotation and its 180° "flips" — you will meet them in
Section 6. Lesson: **use $c'$ to propose candidates, and the phases to choose.**

---
## 5 · All angles at once: an exact fast rotational search in 2D

Each $c_i(\alpha)=\sum_\xi p_i(\xi)\,q_i(R_\alpha^{-1}\xi)$ is a **rotational correlation**. Evaluating it angle by angle
(rotate $q_i$, multiply, sum) costs one full pass over the image per angle. The trick, which the paper uses in 3D (following
Kovacs & Wriggers 2002), is easiest to see in 2D:

1. Resample the centred arrays $p_i$, $q_i$ on **polar rings**: radius $r$ and angle $\varphi$, $P(r,\varphi)$, $Q(r,\varphi)$.
2. A rotation by α is just a **cyclic shift along φ**: $(\Lambda_\alpha q)(r,\varphi)=Q(r,\varphi-\alpha)$.
3. So, ring by ring, $\sum_\varphi P(r,\varphi)\,Q(r,\varphi-\alpha)$ is a 1D **circular cross-correlation** — and by the
   correlation theorem (Module 03) it equals $\mathcal F^{-1}_\varphi\big[\hat P_r(m)\,\overline{\hat Q_r(m)}\big](\alpha)$ for
   **all α at once**, where $\hat P_r(m)$ is the 1D FFT of ring $r$ along φ.
4. Sum over rings with the area weight $r$ (rings get longer as $r$ grows). Because the inverse FFT is linear, sum the
   products first and do **one** inverse FFT:

$$c_i(\alpha)=\mathcal F^{-1}_{m\to\alpha}\Big[\sum_r r\,\hat P_{i,r}(m)\,\overline{\hat Q_{i,r}(m)}\Big].$$

*In words:* turn rotations into shifts by writing the spectra in polar coordinates; shifts are what FFTs do best.

| 2D (this section) | 3D (the paper) |
|---|---|
| polar rings of radius $r$ | spherical shells of radius $r$ |
| Fourier series along a ring, $e^{im\varphi}$ | spherical harmonics on a shell, $Y_l^m(\theta,\phi)$ |
| rotation = shift along φ | rotation mixes the $Y_l^m$ of each degree $l$ via Wigner $D$-matrices |
| one 1D inverse FFT over α | one 3D inverse FFT over three Euler-like angles, using $d^l_{mn}(\pi/2)$ |
| sum over rings with weight $r$ | integral over shells with weight $r^2$ |

In [ ]:
# ── 5.1 Polar resampling of centred Fourier arrays ──
N_PHI = 360                                             # 1° angular sampling
R_RINGS = np.arange(1, int(0.25 * N2) + 1)               # rings up to the same band as before (0.25 cycles/pixel)
PHI = np.arange(N_PHI) * 2 * np.pi / N_PHI
_rr, _pp = np.meshgrid(R_RINGS, PHI, indexing="ij")
POLAR_COORDS = np.array([N2 // 2 + _rr * np.sin(_pp), N2 // 2 + _rr * np.cos(_pp)])   # (row = z, col = x)


def to_polar(a_centred):
    """Sample a centred (fftshifted) 2D Fourier array on rings: returns P[r, φ] (linear interpolation)."""
    return ndimage.map_coordinates(a_centred, POLAR_COORDS, order=1)


def pq_arrays(f, Mf, g, Mg):
    """The p_i and q_i arrays of the table above, centred (fftshift) and with DC removed."""
    f, g = np.asarray(f, np.float64), np.asarray(g, np.float64)       # double precision (NumPy 2 FFTs keep float32)
    Af, Ag = np.fft.fftshift(np.abs(np.fft.fftn(f))), np.fft.fftshift(np.abs(np.fft.fftn(g)))
    mf, mg = np.fft.fftshift(Mf), np.fft.fftshift(Mg)
    c = tuple(n // 2 for n in f.shape)
    Af[c] = Ag[c] = 0
    return [Af * mf, Af**2 * mf, mf], [Ag * mg, mg, Ag**2 * mg]


p_list, q_list = pq_arrays(f_n, Mf_n, g_n, Mf_n)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.9), gridspec_kw=dict(width_ratios=[1, 1, 1.9]))
axes[0].imshow(np.log1p(p_list[0]), cmap="magma"); axes[0].set_title("p₀ = |F f|·M_f (centred, log)")
axes[1].imshow(np.log1p(q_list[0]), cmap="magma"); axes[1].set_title("q₀ = |F g|·M_g (centred, log)")
axes[2].imshow(np.log1p(to_polar(q_list[0])), cmap="magma", aspect="auto", extent=(0, 360, R_RINGS[-1], R_RINGS[0]))
axes[2].set_xlabel("ring angle φ (°)"); axes[2].set_ylabel("ring radius r (Fourier pixels)")
axes[2].set_title("q₀ on polar rings: a rotation of g = a horizontal cyclic shift")
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# ── 5.2 Rotational correlation by FFT, checked against brute force ──
def rotational_correlation_fft(P, Q):
    """c(α) = Σ_r r Σ_φ P(r, φ) Q(r, φ − α) for all α on the φ grid, with one inverse FFT."""
    S = np.sum(R_RINGS[:, None] * np.fft.fft(P, axis=1) * np.conj(np.fft.fft(Q, axis=1)), axis=0)
    return np.real(np.fft.ifft(S))


def rotational_correlation_brute_polar(P, Q):
    """Same quantity by explicitly shifting Q along φ for every α (slow, for checking)."""
    return np.array([np.sum(R_RINGS[:, None] * P * np.roll(Q, j, axis=1)) for j in range(P.shape[1])])


def cprime_all_angles_2d(f, Mf, g, Mg):
    """c'(α) for α = 0, 1, …, 359° via polar rings + FFT (the 2D version of the paper's fast rotational search)."""
    ps, qs = pq_arrays(f, Mf, g, Mg)
    c0, c1, c2 = [rotational_correlation_fft(to_polar(p), to_polar(q)) for p, q in zip(ps, qs)]
    return c0 / np.sqrt(c1 * c2)


P0, Q0 = to_polar(p_list[0]), to_polar(q_list[0])
t0 = time.time(); c_fft = rotational_correlation_fft(P0, Q0); t_fft = time.time() - t0
t0 = time.time(); c_brute = rotational_correlation_brute_polar(P0, Q0); t_brute = time.time() - t0
rel = np.abs(c_fft - c_brute).max() / np.abs(c_brute).max()
print(f"FFT vs brute force on the polar grid: max relative difference {rel:.1e}; "
      f"time {1000 * t_fft:.2f} ms vs {1000 * t_brute:.1f} ms")
assert rel < 1e-9, "the FFT rotational correlation must equal brute force"


def rotate_about_centre(a, angle):
    """Rotate a centred Fourier array about its DC pixel (index N/2) — the Cartesian brute-force alternative."""
    R = rot2d(angle)[::-1, ::-1]                       # (x, z) → (row, col) order
    c = np.array(a.shape) // 2
    return ndimage.affine_transform(a, R.T, offset=c - R.T @ c, order=1)


t0 = time.time()
cp_cart = []
for a in np.radians(np.arange(N_PHI)):
    c0, c1, c2 = [np.sum(p * rotate_about_centre(q, a)) for p, q in zip(p_list, q_list)]
    cp_cart.append(c0 / np.sqrt(c1 * c2))
t_cart = time.time() - t0
t0 = time.time(); cp_fast = cprime_all_angles_2d(f_n, Mf_n, g_n, Mf_n); t_fast = time.time() - t0
print(f"c'(α) for 360 angles: polar+FFT {1000 * t_fast:.1f} ms, Cartesian rotate-and-sum {1000 * t_cart:.0f} ms "
      f"(×{t_cart / t_fast:.0f}); best angle mod 180°: {np.argmax(cp_fast) % 180}° vs {np.argmax(cp_cart) % 180}° "
      f"(true: {TRUE_2D % 180:.0f}°)")
fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.plot(np.arange(N_PHI), cp_cart, color="0.5", lw=3, label="Cartesian brute force (rotate q, multiply, sum)")
ax.plot(np.arange(N_PHI), cp_fast, color="tab:orange", lw=1.2, label="polar rings + one FFT")
ax.set_xlabel("α (°)"); ax.set_ylabel("c'(α)"); ax.legend(fontsize=7); ax.set_title("Same curve, a fraction of the work")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On the polar grid, the FFT result equals the explicit shift-and-sum to machine precision — it is
not an approximation, just a faster way to compute the same sums. Compared with rotating the Cartesian arrays angle by
angle, the curve has the same shape and peak (small differences come from the two different interpolation grids), and is
computed far faster. In 2D the saving is modest in absolute terms; in 3D it is decisive: an exhaustive search costs
(number of rotations) × (volume size), while the spherical-harmonic version costs roughly one expansion of each volume plus
one 3D FFT over the angles.

### 5.3 The complete 2D fast aligner

Following the paper: (1) compute $c'(\alpha)$ for all angles; (2) take its strongest **local maxima** as candidates — and,
because of the 180° ambiguity, also each candidate + 180°; (3) for each candidate, rotate $g$ and run the FFT translation
search with the full constrained correlation; (4) keep the best $(\alpha,\mathbf a)$. One practical detail: the
magnitude score is broad and its peaks can sit 10–15° away from the truth (it ignores phases, and the masks are
interpolated on the rings). So each candidate is scored together with a small neighbourhood (±6°, ±12°), and the winner
is **polished**: the full score is re-evaluated at ±8° around it in 2° steps, then ±2° in 0.5° steps. That is still only
a few dozen evaluations of the expensive score instead of one per angle.

In [ ]:
# ── 5.3 Fast vs brute-force constrained alignment in 2D ──
def full_score_2d(f, Mf, g, a_deg, tilt=60):
    """Constrained correlation (all shifts) after rotating g by a_deg degrees: (score, shift[z, x])."""
    return constrained_correlation_shifts(f, Mf, rotate_image(g, np.radians(a_deg)),
                                          wedge_mask_rotated(f.shape, rot2d(np.radians(a_deg)), tilt), BAND2)


def align_fast_2d(f, Mf, g, Mg, tilt=60, n_cand=4, spread=(-12, -6, 0, 6, 12), polish=(2.0, 0.5)):
    """c'(α) for all angles → top local maxima (and +180°), each with a small neighbourhood → full constrained score
    → short local polishing of the winner. Returns (score, α°, shift)."""
    cp = cprime_all_angles_2d(f, Mf, g, Mg)
    is_peak = (cp >= np.roll(cp, 1)) & (cp >= np.roll(cp, -1))
    cands = np.argsort(-np.where(is_peak, cp, -np.inf))[:n_cand]
    a_list = sorted(set((int(c) + h + s) % 360 for c in cands for h in (0, 180) for s in spread))
    best = max(((*full_score_2d(f, Mf, g, a, tilt), a) for a in a_list), key=lambda x: x[0])
    for step in polish:                                   # polish around the winner, as in 3D (Section 6)
        best = max(((*full_score_2d(f, Mf, g, best[2] + d, tilt), (best[2] + d) % 360)
                    for d in step * np.arange(-4, 5)), key=lambda x: x[0])
    return best[0], best[2], best[1]


def align_brute_2d(f, Mf, g, tilt=60, step=2, constrained=True):
    """Try every angle on a grid; FFT translation search for each. Returns (score, α°, shift)."""
    best = (-np.inf, 0, None)
    for a in np.arange(0, 360, step):
        omega = wedge_mask_rotated(f.shape, rot2d(np.radians(a)), tilt) * Mf if constrained else np.ones_like(Mf)
        s, sh = corr_all_shifts(f, rotate_image(g, np.radians(a)), omega * BAND2)
        if s > best[0]:
            best = (s, a, sh)
    return best


t0 = time.time(); b_fast = align_fast_2d(f_n, Mf_n, g_n, Mf_n); t_f = time.time() - t0
t0 = time.time(); b_brute = align_brute_2d(f_n, Mf_n, g_n, step=1); t_b = time.time() - t0
print(f"fast aligner:  α = {b_fast[1]:.1f}°, shift (z, x) = {b_fast[2]}, score {b_fast[0]:.3f}, {1000 * t_f:.0f} ms")
print(f"brute force:   α = {b_brute[1]}°, shift (z, x) = {b_brute[2]}, score {b_brute[0]:.3f}, {1000 * t_b:.0f} ms "
      f"(×{t_b / t_f:.0f} slower)")
a_exp = (np.array([2, -1]) - rot2d(np.radians(TRUE_2D)) @ np.array([-1, 3]))[::-1]   # t_f − R t_g, (z, x) order
print(f"truth:         α = {TRUE_2D:.0f}°, shift (z, x) = {np.round(a_exp, 1)}")

Now a small benchmark: 8 random pairs at each of four noise levels, aligned three ways — plain-correlation brute force,
constrained brute force (2° steps), and the fast aligner. A trial counts as a success if the angular error is below 10°.

In [ ]:
# ── 5.4 Accuracy and speed vs SNR (2D) ──
SNRS_2D = [None, 1.0, 0.3, 0.1]
methods = {"plain, brute force": lambda f, M, g: align_brute_2d(f, M, g, step=3, constrained=False),
           "constrained, brute force": lambda f, M, g: align_brute_2d(f, M, g, step=3),
           "constrained, fast (c' + FFT)": lambda f, M, g: align_fast_2d(f, M, g, M)}
bench = {m: {"succ": [], "med": [], "ms": []} for m in methods}
rng_b = np.random.default_rng(21)
t_all = time.time()
for snr in SNRS_2D:
    errs = {m: [] for m in methods}; tms = {m: 0.0 for m in methods}
    for _ in range(8):
        a_f, a_g = rng_b.uniform(0, 2 * np.pi, 2)
        f, M = observe_2d(a_f, rng_b.uniform(-3, 3, 2), snr, 60, rng_b)
        g, _ = observe_2d(a_g, rng_b.uniform(-3, 3, 2), snr, 60, rng_b)
        for m, fn in methods.items():
            t0 = time.time(); res = fn(f, M, g); tms[m] += time.time() - t0
            errs[m].append(ang_err(res[1], np.degrees(a_f - a_g)))
    for m in methods:
        bench[m]["succ"].append(np.mean(np.array(errs[m]) < 10)); bench[m]["med"].append(np.median(errs[m]))
        bench[m]["ms"].append(1000 * tms[m] / 8)
    print(f"SNR {str(snr):>4s}: " + " | ".join(f"{m}: {100 * bench[m]['succ'][-1]:.0f} % ok, median {bench[m]['med'][-1]:.0f}°"
                                              for m in methods))
print(f"benchmark time {time.time() - t_all:.0f} s")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
labels = ["∞" if s is None else str(s) for s in SNRS_2D]
for (m, col) in zip(methods, ["tab:gray", "tab:blue", "tab:orange"]):
    axes[0].plot(labels, 100 * np.array(bench[m]["succ"]), "-o", color=col, label=m)
axes[0].set_xlabel("SNR"); axes[0].set_ylabel("success rate (%)"); axes[0].set_ylim(-5, 105); axes[0].legend(fontsize=7)
axes[0].set_title("2D alignment accuracy (error < 10°)")
axes[1].bar(range(3), [np.mean(bench[m]["ms"]) for m in methods], color=["tab:gray", "tab:blue", "tab:orange"])
axes[1].set_xticks(range(3)); axes[1].set_xticklabels([m.replace(", ", "\n") for m in methods], fontsize=7)
axes[1].set_ylabel("time per alignment (ms)"); axes[1].set_title("speed")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Plain correlation fails even at infinite SNR (the wedge bias of Section 2). Constrained
correlation fixes accuracy; the fast aligner keeps (roughly) that accuracy while evaluating the full score at a few
dozen angles instead of all of them. In 2D that makes it only a few times faster, because brute force is cheap here
anyway; in 3D (Section 6) the same idea saves orders of magnitude. All methods degrade as noise grows: a 64 × 64 slice
simply contains too little signal at SNR ≈ 0.1 (3D volumes contain far more voxels, and tolerate more noise). With 8
pairs per condition, differences of one or two successes are within chance.

✏️ **Your turn.** In `align_fast_2d` set `n_cand=1`. How often does the fast aligner now fail at SNR 1, and why?
<details><summary>Answer</summary>The highest c′ peak is not always the true angle once noise is present (c′ is only an
upper bound and is blind to phases); with a single candidate, a wrong peak cannot be corrected by the phase-sensitive
translation step. Several candidates make the method robust.</details>

---
## 6 · Fast alignment in 3D

### 6.1 How the paper searches all 3D rotations at once (explained, not implemented)

In 3D, rings become **spherical shells** $|\xi|=r$ in Fourier space. On each shell, a function such as $p_i$ is expanded
in **spherical harmonics** — the 3D analogue of the Fourier series on a circle:

$$p_i(r,\theta,\phi)\approx\sum_{l=0}^{B}\sum_{m=-l}^{l}C^{p_i}_{lm}(r)\,Y_l^m(\theta,\phi).$$

Rotating a function on a sphere does not mix harmonics of different degree $l$; within a degree it mixes the $2l+1$
coefficients through **Wigner $D$-matrices**. Writing each rotation as a product of rotations about fixed axes, Kovacs &
Wriggers (2002) showed that the rotational correlation for **all** rotations becomes a 3D inverse FFT over three angles,
with only constant matrices $d^l_{mh}(\pi/2)$ in between. Xu, Beck & Alber (2012) use exactly this on Fourier shells
(their eq. 10, schematic):

$$c_{R}=\mathcal F^{-1}_{m,h,m'}\Big[\sum_l d^l_{mh}(\tfrac\pi2)\,d^l_{hm'}(\tfrac\pi2)\int C^{p}_{lm}(r)\,\overline{C^{q}_{lm'}(r)}\,r^2dr\Big].$$

*In words:* expand each shell once, multiply coefficient matrices and integrate over radius, then one 3D FFT gives the
score at every rotation of a $2B\times2B\times2B$ angular grid (angular step $180°/B$). In AITom the bandwidth is
`L = 36` by default and the C++ routine `rot_search_cor` is called three times — once per $(p_i,q_i)$ pair:

```text
cors_12     = rot_search_cor(|F1| m1², |F2| m2²)         # c0
sqt_cors_11 = sqrt(rot_search_cor(|F1|² m1², m2²))        # √c1
sqt_cors_22 = sqrt(rot_search_cor(m1², |F2|² m2²))        # √c2
cors = cors_12 / (sqt_cors_11 * sqt_cors_22)             # c'(R) for all rotations
```

followed by `local_max_angles` (candidates) and `cons_corr_max` (FFT translation search with the constrained correlation)
for each candidate. Implementing spherical harmonics and Wigner matrices is beyond this module (see 🚀 Bonus).

### 6.2 Our 3D version: a rotation grid scored by $c'$

We keep everything else identical but replace the all-rotations FFT by a **coarse rotation grid** (20°, ≈ 1,900
rotations). For each grid rotation we evaluate $c_0,c_1,c_2$ by sampling $q_i$ at the rotated frequencies $R^{-1}\xi$ —
only on the frequencies where $f$ was measured (where $p_i\neq0$), and with the rotated wedge of $g$ computed exactly. We
then keep the 20 best, mutually distinct (> 20° apart) rotations as candidates, score each with the full constrained
correlation over all shifts, and finally polish the winner on two small local grids (8° and 4° steps), again with the
full score. To keep both scores focused on where the molecule has signal, we use Fourier radii up to 12 voxels for
$c'$ and frequencies up to 0.25 cycles/voxel (1/(4 nm)) for $c$.

In [ ]:
# ── 6.2 Building blocks of the 3D aligner ──
N3 = 32
K3 = radial_frequency((N3,) * 3)
BAND3 = (K3 <= 0.25).astype(np.float32)
_g = np.arange(N3) - N3 // 2
KZc, KYc, KXc = np.meshgrid(_g, _g, _g, indexing="ij")              # centred integer frequency coordinates
RADc = np.sqrt(KXc**2 + KYc**2 + KZc**2)


def cprime_scorer(f, Mf, g, tilt_g, r_min=1, r_max=12):
    """Return a function rotations → c'(R) (eqs. 8–9) for the pair (f, g), by sampling g's centred magnitude spectrum at
    Rᵀξ for every frequency ξ where f was measured (r_min ≤ |ξ| ≤ r_max)."""
    Af = np.fft.fftshift(np.abs(np.fft.fftn(f)))
    Ag = np.fft.fftshift(np.abs(np.fft.fftn(g)))
    sel = (np.fft.fftshift(Mf) > 0) & (RADc >= r_min) & (RADc <= r_max)
    xi = np.stack([KXc[sel], KYc[sel], KZc[sel]]).astype(float)        # (3, n_freq) in (x, y, z) order
    p0, p1 = Af[sel], Af[sel] ** 2
    tan_w = np.tan(np.radians(tilt_g))
    c = N3 // 2

    def score(rotations, batch=200):
        out = np.empty(len(rotations))
        for b in range(0, len(rotations), batch):
            Rb = rotations[b:b + batch]
            eta = np.einsum("rji,jp->rip", Rb, xi)                        # Rᵀξ for every rotation in the batch
            Wg = (np.abs(eta[:, 2]) <= np.abs(eta[:, 0]) * tan_w + 1e-9)  # g's wedge at Rᵀξ, exactly
            coords = np.stack([eta[:, 2] + c, eta[:, 1] + c, eta[:, 0] + c]).reshape(3, -1)
            q = ndimage.map_coordinates(Ag, coords, order=1).reshape(len(Rb), -1) * Wg   # |F g|(Rᵀξ)·M_g(Rᵀξ)
            c0 = np.sum(p0 * q, axis=1)                                   # Σ p0 · Λ_R q0
            c1 = np.sum(p1 * Wg, axis=1)                                  # Σ p1 · Λ_R q1
            c2 = np.sum(q ** 2, axis=1)                                   # Σ p2 · Λ_R q2 (|F g|² from the same samples)
            out[b:b + batch] = c0 / np.sqrt(c1 * c2 + 1e-30)
        return out
    return score


def rotation_nms(rotations, scores, k, min_deg):
    """Indices of the k best rotations that are pairwise more than min_deg apart (non-maximum suppression)."""
    keep = []
    for i in np.argsort(-scores):
        if all(rotation_angle_deg(rotations[i], rotations[j]) > min_deg for j in keep):
            keep.append(i)
            if len(keep) == k:
                break
    return keep


def local_grid(R0, step_deg, half=1):
    """Rotations R = exp(ω) R0 for ω on a (2·half+1)³ cube of rotation vectors with the given step."""
    a = np.radians(np.arange(-half, half + 1) * step_deg)
    w = np.stack(np.meshgrid(a, a, a, indexing="ij"), -1).reshape(-1, 3)
    return Rotation.from_rotvec(w).as_matrix() @ R0


def full_score(f, Mf, g, R, tilt_g):
    """Constrained correlation over all shifts for rotation R: (score, shift[z, y, x])."""
    return constrained_correlation_shifts(f, Mf, rotate_volume(g, R), wedge_mask_rotated(f.shape, R, tilt_g), BAND3)


def align_fast_3d(f, Mf, g, tilt_g, grid, n_cand=20, refine_steps=(8, 4)):
    """Grid of c' → candidates → full constrained correlation (+ shifts) → local polishing. Returns a dict."""
    T = {}
    t0 = time.time(); score = cprime_scorer(f, Mf, g, tilt_g); cp = score(grid); T["c' on grid"] = time.time() - t0
    t0 = time.time()
    cand = rotation_nms(grid, cp, n_cand, 20)
    results = [(full_score(f, Mf, g, grid[i], tilt_g), grid[i]) for i in cand]
    (s, sh), R = max(results, key=lambda x: x[0][0])
    T["candidates (phases)"] = time.time() - t0
    t0 = time.time()
    for step in refine_steps:
        (s, sh), R = max([(full_score(f, Mf, g, Rl, tilt_g), Rl) for Rl in local_grid(R, step)], key=lambda x: x[0][0])
    T["local polishing"] = time.time() - t0
    return dict(R=R, shift=sh, score=s, cprime=cp, cand=cand, times=T)


GRID20 = euler_grid(20)
print(f"20° grid: {len(GRID20)} rotations")

### 6.3 Test particles

Like the paper, we first test on an **asymmetric Gaussian phantom** — four blobs of different sizes and shapes, a shape
with unambiguous orientation — and then on the ribosome. The ribosome is harder than it looks: at ~3–4 nm resolution its
outline is close to an ellipsoid, so its orientation is encoded only in subtle features.

In [ ]:
# ── 6.3 The asymmetric phantom and the ribosome ──
def asymmetric_phantom(n=N3):
    """Four anisotropic Gaussian blobs at different positions (an asymmetric test object, cf. the paper's phantoms)."""
    g = np.arange(n) - (n - 1) / 2
    Z, Y, X = np.meshgrid(g, g, g, indexing="ij")
    blobs = [((0, 0, 0), (4.0, 3.0, 2.5), 1.0), ((8, 1, -2), (2.0, 2.0, 2.0), 0.9),
             ((-3, 8, 2), (2.5, 1.5, 1.5), 0.8), ((-2, -4, 8), (1.5, 1.5, 3.0), 0.7)]   # (x, y, z) centre, σ's, weight
    v = sum(w * np.exp(-((X - mx)**2 / (2 * sx**2) + (Y - my)**2 / (2 * sy**2) + (Z - mz)**2 / (2 * sz**2)))
            for (mx, my, mz), (sx, sy, sz), w in blobs)
    return (v / v.max()).astype(np.float32)


PHANTOM = asymmetric_phantom()
fig, axes = plt.subplots(2, 3, figsize=(7.5, 5.0))
show_orthoslices(PHANTOM, "phantom", axes=axes[0])
show_orthoslices(rib32, "ribosome", axes=axes[1])
plt.tight_layout(); plt.show()

In [ ]:
# ── 6.4 A worked example: phantom, SNR 0.3, ±60° ──
rng_w = np.random.default_rng(5)
R_f, R_g = random_rotation(rng_w), random_rotation(rng_w)
t_f, t_g = rng_w.uniform(-2, 2, 3), rng_w.uniform(-2, 2, 3)
f3, Mf3 = observe(PHANTOM, R_f, t_f, 0.3, 60, rng_w)
g3, _ = observe(PHANTOM, R_g, t_g, 0.3, 60, rng_w)
R_true = R_f @ R_g.T
a_true = (t_f - R_true @ t_g)[::-1]                          # expected shift in (z, y, x) order
res = align_fast_3d(f3, Mf3, g3, 60, GRID20)
err_grid = np.array([rotation_angle_deg(R, R_true) for R in GRID20])
print(f"rotation error: {rotation_angle_deg(res['R'], R_true):.1f}° | shift found {res['shift']} vs expected "
      f"{np.round(a_true, 1)} (voxels, z y x) | score {res['score']:.3f}")
print("time: " + ", ".join(f"{k} {v:.2f} s" for k, v in res["times"].items()))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].scatter(err_grid, res["cprime"], s=4, color="0.6", label="all grid rotations")
axes[0].scatter(err_grid[res["cand"]], res["cprime"][res["cand"]], s=30, color="tab:orange", label="20 candidates")
axes[0].set_xlabel("angular distance to the true rotation (°)"); axes[0].set_ylabel("c'(R)")
axes[0].set_title("The c' landscape is nearly flat"); axes[0].legend(fontsize=7)
sl = np.s_[N3 // 2]
smooth = lambda v: ndimage.gaussian_filter(v, 1.0)[sl]                # light smoothing for display only
axes[1].imshow(np.hstack([smooth(f3), smooth(g3), smooth(rotate_volume(g3, res["R"], res["shift"][::-1])),
                          smooth(rotate_volume(PHANTOM, R_f, t_f))]))
axes[1].set_title("xy slices (smoothed): f | g | g aligned onto f | truth", fontsize=8); axes[1].axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The scatter shows the $c'$ landscape, and it is remarkably flat: the grid rotations nearest
the truth (left) score high, but so do many rotations far away, including 150–180° "flips" (the near-mirror ambiguity of
magnitude spectra). The single best $c'$ is therefore not guaranteed to be the truth. That is why the method keeps
**many** candidates (the paper: local maxima, fewer than 100) and lets the phase-sensitive constrained correlation choose
among them. The translation comes out of the same FFT, to the nearest voxel.

### 6.5 Accuracy versus noise and tilt range

In [ ]:
# ── 6.5a Phantom: accuracy vs SNR (±60°) ──
def benchmark_3d(obj, snrs, tilts, n_pairs, seed):
    """Align n_pairs random pairs per (snr, tilt) condition; return {(snr, tilt): list of rotation errors (°)}."""
    rng_l = np.random.default_rng(seed)
    out = {}
    for snr in snrs:
        for tilt in tilts:
            errs = []
            for _ in range(n_pairs):
                Rf, Rg = random_rotation(rng_l), random_rotation(rng_l)
                f, Mf = observe(obj, Rf, rng_l.uniform(-2, 2, 3), snr, tilt, rng_l)
                g, _ = observe(obj, Rg, rng_l.uniform(-2, 2, 3), snr, tilt, rng_l)
                errs.append(rotation_angle_deg(align_fast_3d(f, Mf, g, tilt, GRID20)["R"], Rf @ Rg.T))
            out[(snr, tilt)] = np.array(errs)
            print(f"  SNR {str(snr):>4s}, ±{tilt}°: errors {np.round(errs).astype(int)}")
    return out


t0 = time.time()
acc_snr = benchmark_3d(PHANTOM, [None, 0.3, 0.1, 0.03], [60], n_pairs=4, seed=31)
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
# ── 6.5b Phantom vs tilt range (SNR 0.3), and the ribosome ──
t0 = time.time()
acc_tilt = benchmark_3d(PHANTOM, [0.3], [40, 80], n_pairs=4, seed=32)
acc_tilt[(0.3, 60)] = acc_snr[(0.3, 60)]
print("ribosome:")
acc_rib = benchmark_3d(rib32, [None, 0.3], [60], n_pairs=4, seed=33)
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
# ── 6.5c Summary plot ──
def success(errs, thr=15):
    """Fraction of alignments with rotation error below thr degrees."""
    return float(np.mean(np.asarray(errs) < thr))


fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
snr_labels = ["∞", "0.3", "0.1", "0.03"]
axes[0].bar(snr_labels, [100 * success(acc_snr[(s, 60)]) for s in [None, 0.3, 0.1, 0.03]], color="tab:orange")
axes[0].set_xlabel("SNR"); axes[0].set_ylabel("success (%) — error < 15°"); axes[0].set_title("phantom, ±60°")
axes[1].bar(["±40°", "±60°", "±80°"], [100 * success(acc_tilt[(0.3, t)]) for t in (40, 60, 80)], color="tab:purple")
axes[1].set_xlabel("tilt range"); axes[1].set_title("phantom, SNR 0.3")
axes[2].bar(["∞", "0.3"], [100 * success(acc_rib[(s, 60)]) for s in (None, 0.3)], color="tab:red")
axes[2].set_xlabel("SNR"); axes[2].set_title("ribosome, ±60°")
for ax in axes:
    ax.set_ylim(0, 105)
plt.tight_layout(); plt.show()
all_err = np.concatenate([v for v in acc_snr.values()] + [v for v in acc_tilt.values()] + [v for v in acc_rib.values()])
flips = all_err[all_err > 15]
print(f"successful alignments: median error {np.median(all_err[all_err <= 15]):.1f}° "
      f"(the 20° grid was polished with 8° and 4° steps)")
print(f"failures: {len(flips)}; of these, {np.sum(flips > 140)} are 140–180° 'flips'")

🔍 **What just happened?** With low noise the aligner recovers the rotation to a few degrees — the limit set by our 4°
polishing step (Module 13 refines further). As noise increases, failures appear; some of them are **flips**: rotations
140–180° away that put the molecule's overall outline in the right place but its asymmetric details on the wrong side. A
narrower tilt range (±40°) removes much more information and the search collapses; ±80° performs about like ±60° (with
only 4 pairs per condition, one failure more or less is chance). The roundish ribosome is harder than the strongly
asymmetric phantom: one of its four noise-free pairs already ends tens of degrees off, and at SNR 0.3 one pair is
flipped, whereas the phantom aligned every pair at both settings. With four random pairs per condition, *which*
rotations were drawn matters as much as the noise level.

⚠️ **Do not compare our SNR numbers with the paper's.** The paper reports accurate alignment down to SNR 0.01–0.005 on
64³ phantoms simulated through a full tilt-series model; we use 32³ volumes (8× fewer voxels, so far less signal per
subtomogram), a simple noise model, and a coarse grid instead of the exact SH-FFT search.

### 6.6 Speed

In [ ]:
# ── 6.6 Speed: our fast 3D aligner vs brute force at the same final precision ──
t0 = time.time()
for _ in range(5):
    full_score(f3, Mf3, g3, random_rotation(rng), 60)
t_eval = (time.time() - t0) / 5
t_fast = sum(res["times"].values())
n4 = euler_grid_size(4)
print(f"one full-score evaluation (rotate + masks + FFT shifts) at 32³: {1000 * t_eval:.1f} ms")
print(f"fast aligner (grid c' + {len(res['cand'])} candidates + polishing): {t_fast:.2f} s per pair")
print(f"brute force on a 4° grid ({n4:,d} rotations): ≈ {n4 * t_eval / 60:.0f} min per pair "
      f"→ speed-up ≈ ×{n4 * t_eval / t_fast:,.0f}")
print("paper (64³, 2.5° sampling, MATLAB, 2.67 GHz): 153 s vs ≈ 9.67 × 10⁵ s for exhaustive scanning (≈ ×6,000)")

🔍 **What just happened?** Scoring one rotation with the full constrained correlation costs a few milliseconds; a
brute-force search at 4° needs over 100,000 of them. Our aligner spends most of its time on the cheap magnitude score and
evaluates the expensive score only ~70 times, so it is two to three orders of magnitude faster — the same logic that made
the paper's method ≈ 6,000× faster than exhaustive scanning (theirs is faster still, because the spherical-harmonic FFT
scores every rotation of a fine grid at once).

---
### 📄 Paper connection — Xu, Beck & Alber (2012) — the centrepiece of this module

**Paper:** Xu M, Beck M, Alber F. *High-throughput subtomogram alignment and classification by Fourier space constrained
fast volumetric matching.* J Struct Biol 178(2):152–164 (2012). [doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014)
(open-access author manuscript: PMC3821800).

- **Question it answers:** how can thousands of noisy, missing-wedge subtomograms be aligned (and classified without a
  reference) accurately and fast enough?
- **What the paper actually does:** (1) **constrained correlation** — compare $f$ and $\tau_{\mathbf a}\Lambda_Rg$ only on
  $\Omega=M_f\cdot\Lambda_RM_g$, mean-normalized (eqs. 1–3), rewritten in Fourier space with the DC term removed;
  (2) **drop the phases** — the magnitude score $c'(R)=c_0/\sqrt{c_1c_2}$ is translation-invariant and an upper bound of
  $c$ over all shifts (eqs. 8–9); (3) **all rotations at once** — each $c_i$ is a rotational correlation computed by
  expanding Fourier shells in spherical harmonics, integrating over radius, and one inverse 3D FFT over Euler angles
  with Wigner $d^l(\pi/2)$ (the Kovacs & Wriggers approach; eq. 10); (4) **candidates** — local maxima of $c'$ (fewer than
  100), each followed by an FFT translational search on the full $c$. The paper also builds a reference-free
  **classification** scheme on top (pairwise scores → hierarchical clustering → class averages, with feature-enhancement
  weights) — Module 14.
- **What our toy keeps:** eqs. 1–3 exactly; the $p_i/q_i$ decomposition of $c'$; the exact FFT-over-angles idea (in 2D);
  candidates + phase-based translation search; accuracy-vs-SNR/tilt and speed experiments.
- **What our toy leaves out:** spherical harmonics and Wigner matrices (we use a 20° grid + local polishing in 3D);
  64³ volumes and the tilt-series simulation; the classification part.
- **Notes on the equations:** eqs. 1–3 are clean in the open-access text; the Fourier-space forms (eqs. 7–9) render poorly
  online and were reconstructed with the help of the AITom implementation, which matches them term by term.
- **Result reported:** the authors report 153 s per alignment of 64³ volumes at 2.5° angular sampling versus about
  9.67 × 10⁵ s (more than 10 days) for exhaustive scanning (≈ 6,000× faster); accurate alignment at SNR 0.01 and 0.005
  for typical tilt ranges, failing at 0.001 (rotation error $\Delta E=\lVert R_{true}^TR_{align}-I\rVert_F>0.1$ counted as
  failure); and classification accuracies of 100 % (4 simulated complexes), 68.2 % (786 experimental GroEL/GroEL–GroES
  subtomograms) and 99 % (*Leptospira interrogans* ribosomes vs ATP synthase).
- **Code:** AITom `aitom/align/fast/util.py` (`align_vols`, `fast_rotation_align`) → C++ core
  `aitom/tomominer/core/src/align.cpp` (`combined_search`, `rot_search_cor`, `local_max_angles`, `cons_corr_max`).

🧮 **The paper's error measure.** $\Delta E=\lVert R_{true}^TR-I\rVert_F$. For a rotation by angle θ,
$\lVert R-I\rVert_F^2=\operatorname{tr}\big((R-I)^T(R-I)\big)=6-2\operatorname{tr}R=4-4\cos\theta$, so
$\Delta E=2\sqrt2\,|\sin(\theta/2)|$; the failure threshold $\Delta E=0.1$ corresponds to θ ≈ 2.0°.

### 📄 Cited foundations (not Xu-lab papers)

- **Förster F, Pruggnaller S, Seybert A, Frangakis AS (2008).** *Classification of cryo-electron sub-tomograms using
  constrained correlation.* J Struct Biol 161:276–286. [doi:10.1016/j.jsb.2007.07.006](https://doi.org/10.1016/j.jsb.2007.07.006)
  — introduced the idea of correlating subtomograms only over their commonly observed Fourier region; Xu et al. (2012)
  build on it.
- **Kovacs JA, Wriggers W (2002).** *Fast rotational matching.* Acta Cryst D 58:1282–1286.
  [doi:10.1107/S0907444902009794](https://doi.org/10.1107/S0907444902009794) — the spherical-harmonic + FFT method for
  scoring all rotations at once, which Xu et al. adapt to Fourier-space shells.

---
## ✅ Checkpoint

1. Why does plain correlation of two missing-wedge subtomograms favour rotations that align the wedges?
   <details><summary>Answer</summary>The missing wedge creates the same kind of artefact (elongation along the beam,
   missing directions) in both volumes, fixed in the microscope frame. Superimposing the wedges superimposes these shared
   artefacts, which can raise the correlation more than superimposing the molecules.</details>
2. Write Ω for constrained correlation and say in words what it does.
   <details><summary>Answer</summary>Ω = M_f · Λ_R M_g: the set of Fourier coefficients measured in f AND in the rotated
   g. Both volumes are filtered to Ω before the (mean-normalized) correlation, so neither wedge can bias the score.</details>
3. Why can $c'(R)$ be computed without knowing the translation?
   <details><summary>Answer</summary>A translation multiplies Fourier coefficients by phase factors of modulus 1, so the
   magnitudes |F g| — the only thing c′ uses — do not change.</details>
4. In 2D, why does $c'(\alpha)$ always have a twin peak at α + 180°, and how does the aligner resolve it?
   <details><summary>Answer</summary>Magnitude spectra of real images satisfy |F(ξ)| = |F(−ξ)|, and ξ → −ξ is a 180°
   rotation in 2D. The aligner evaluates both α and α + 180° with the full constrained correlation, whose phases differ.</details>
5. How does the polar FFT turn "all rotations" into one operation?
   <details><summary>Answer</summary>On polar rings a rotation is a cyclic shift along the angle; the correlation over all
   shifts is one FFT product + inverse FFT (correlation theorem); summing over rings before the inverse FFT gives c_i(α)
   for all α at once.</details>
6. A rotation error of 3° — what is ΔE?
   <details><summary>Answer</summary>ΔE = 2√2·sin(1.5°) ≈ 0.074, just below the paper's failure threshold of 0.1.</details>

## 🧠 Questions & discussion

1. Constrained correlation compares fewer coefficients when the two wedges overlap poorly. What does that do to the
   *statistical reliability* of the score, and when could it become a problem?
2. Our 3D aligner keeps 20 candidates. What is the trade-off in choosing that number? How would you choose it on real data?
3. The fast search scores rotations with magnitudes only. Can you construct a molecule for which $c'$ is *exactly* the same
   for two different rotations that are not related by the molecule's symmetry? (Hint: think of mirror images.)
4. Low-resolution ribosomes failed more often than the phantom. What does that suggest about aligning small or roundish
   complexes in cells, and what extra information could help?
5. The paper aligns all pairs of subtomograms for reference-free classification. How does the number of alignments grow
   with the number of subtomograms, and why does that make speed essential?

## 🚀 Bonus — going further

- **Spherical harmonics yourself.** Expand one Fourier shell of `rib32` with `scipy.special.sph_harm_y` and verify that the
  energy per degree $l$, $\sum_m|C_{lm}|^2$, does not change when the volume is rotated (a link back to Module 02's
  rotation-invariant features).
- **Official code.** Read `aitom/tomominer/core/src/align.cpp` in [AITom](https://github.com/xulabs/aitom): map
  `rot_search_cor` → $c_i$, the `r2` factor → the $r^2dr$ shell integral, `local_max_angles` → candidates,
  `cons_corr_max` → our `constrained_correlation_shifts`.
- **Weighted masks.** Real wedges are not perfectly binary (dose weighting, CTF). Replace the binary masks with smooth
  weights and use the squared-mask formulas.
- **Finer grid.** Replace `GRID20` by `euler_grid(15)` and compare accuracy and time.
- **Deep dive tutorial:** [`../../foundation/SubtomogramAveraging`](../../foundation/SubtomogramAveraging/).

## 📝 Lab note

```text
Module 12 — date:
Input: 2D ribosome slab + 3D phantom / ribosome (32³), rotated, ±60° wedge, noise
Output: rotation (+ shift) aligning g onto f; success rates vs SNR and tilt; speed-ups
Method: constrained correlation (Ω = M_f·Λ_R M_g, eqs. 1–3); c'(R) = c0/√(c1 c2) from magnitudes; polar-FFT (2D) /
        grid (3D) rotational search → candidates → FFT translation search
Key parameter changed & effect:
Failure mode observed (wedge bias? 180° flips? noise?):
Paper link: Xu, Beck & Alber 2012 (J Struct Biol 178:152)
```

## 📚 References

1. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014
2. Förster F, Pruggnaller S, Seybert A, Frangakis AS. Classification of cryo-electron sub-tomograms using constrained correlation. *J Struct Biol* 161:276–286 (2008). doi:10.1016/j.jsb.2007.07.006
3. Kovacs JA, Wriggers W. Fast rotational matching. *Acta Cryst D* 58:1282–1286 (2002). doi:10.1107/S0907444902009794
4. Zeng X, Xu M. AITom: Open-source AI platform for cryo-electron tomography data analysis. arXiv:1911.03044 (2019); <https://github.com/xulabs/aitom>
5. Frazier Z, Xu M, Alber F. TomoMiner and TomoMinerCloud: a software platform for large-scale subtomogram structural analysis. *Structure* 25:951–961 (2017). doi:10.1016/j.str.2017.04.016

## ➡️ Next

**[13 · Precise alignment, crowded neighbours and scaling up to millions](13_precise_alignment_crowding_scale.ipynb)** —
polish coarse alignments to sub-degree precision with Levenberg–Marquardt, cope with neighbouring molecules inside the box,
and spread thousands of alignments over many CPUs. Course overview: [`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).